# 2. Liquidity Constraints, Fire Sales and Dealer Losses

Compare the unlimited-cash benchmark with limited cash and illiquid assets. Dealer losses are reported unconditionally and conditional on being strictly positive.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.otc_margining import ModelConfig,simulate_prices,simulate_default_days,simulate_liquidity,summarize_simulation
cfg=ModelConfig(paths=20000,horizon=252,maturity=300,mpor=10,seed=42)
prices=simulate_prices(cfg,seed=100,extra_days=cfg.mpor)
default_days=simulate_default_days(cfg,seed=200)
unlimited=simulate_liquidity(prices,default_days,cfg,mode='unlimited')
fire=simulate_liquidity(prices,default_days,cfg,mode='fire_sale')
summary=pd.DataFrame([summarize_simulation(unlimited,cfg),summarize_simulation(fire,cfg)],index=['Unlimited cash','Limited cash + fire sales']).T
summary

## Interpret the dealer loss distribution

The unconditional loss distribution has a point mass at zero. Conditional percentiles describe severity only among paths with a positive dealer loss.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
for result,label in [(unlimited,'Unlimited cash'),(fire,'Fire-sale liquidity')]:
    positive=result['losses'][result['losses']>0]
    if positive.size: ax.hist(positive,bins=40,density=True,alpha=.45,label=label)
ax.set(xlabel='Dealer loss ($), conditional on positive loss',ylabel='Density',title='Conditional dealer loss distribution')
ax.legend(frameon=False);plt.show()
print('Fire-sale path share:',f"{np.mean(fire['fire_sale_events']>0):.2%}")
print('Mean fire-sale cost/path:',f"${fire['fire_sale_cost'].mean():,.2f}")